In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import h5py
from torch.utils.data import Dataset, DataLoader, Subset
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer
from tqdm.auto import tqdm
import evaluate  # pip install evaluate rouge_score
import os

# ==================================================================================
# 1. CONFIGURATION
# ==================================================================================
MODEL_WEIGHTS_PATH = "static-graph-phase-2.pt"  # Or "static-graph-phase-1.pt" depending on your save
H5_FILE_PATH = "/home/poorna/data/eeg_dataset_1400_multilabel.h5"
LOCAL_MODEL_PATH = "/home/poorna/models/bert-base-uncased"

# Use GPU if available for speed, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running Evaluation on: {device}")

# Tokenizer
try:
    tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)
except:
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

PAD_ID = tokenizer.pad_token_id
SOS_ID = tokenizer.cls_token_id
EOS_ID = tokenizer.sep_token_id
TEXT_VOCAB_SIZE = tokenizer.vocab_size
NUM_COLORS = 9       
NUM_OBJECTS = 6      

# ==================================================================================
# 2. MODEL DEFINITIONS (Dense GCN - Must Match Training)
# ==================================================================================

class DenseGCNLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)

    def forward(self, x, adj):
        x = self.linear(x) 
        return torch.einsum('nm, btni -> btmi', adj, x)

class SpatioTemporalEEGEncoder(nn.Module):
    def __init__(self, num_channels=62, enc_hidden=256, num_layers=2, dropout=0.2):
        super().__init__()
        self.num_channels = num_channels
        self.gcn1 = DenseGCNLayer(1, enc_hidden) 
        self.gcn2 = DenseGCNLayer(enc_hidden, enc_hidden)
        self.rnn = nn.GRU(enc_hidden, enc_hidden, num_layers, bidirectional=True, dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, eeg, adj_matrix):
        # eeg: [Batch, Channels, Time] -> [Batch, Time, Channels, 1]
        x = eeg.permute(0, 2, 1).unsqueeze(-1) 
        x = F.relu(self.gcn1(x, adj_matrix))
        x = self.dropout(x)
        x = F.relu(self.gcn2(x, adj_matrix)) 
        x = torch.mean(x, dim=2) # Mean Pool over Channels
        encoder_outputs, encoder_hidden = self.rnn(x)
        return encoder_outputs.permute(1, 0, 2), encoder_hidden

class MetadataEncoder(nn.Module):
    def __init__(self, num_colors, num_objects):
        super().__init__()
        self.color_p = nn.Sequential(nn.Linear(num_colors, 64), nn.ReLU(), nn.Linear(64, 32))
        self.obj_p = nn.Sequential(nn.Linear(num_objects, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, 32))
        self.output_dim = 64
    def forward(self, m): return torch.cat([self.color_p(m[:,:9]), self.obj_p(m[:,9:])], dim=1)

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, enc_hidden, dec_hidden, meta_dim, num_layers, pad_id, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.attention = nn.Linear(enc_hidden * 2, dec_hidden)
        self.rnn = nn.GRU(emb_dim + enc_hidden * 2 + meta_dim + enc_hidden * 2, dec_hidden, num_layers, dropout=dropout)
        self.fc_out = nn.Linear(dec_hidden, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.bridge = nn.Linear(enc_hidden * 2, dec_hidden)
        self.num_layers = num_layers
    
    def init_hidden(self, enc_hid):
        hidden = enc_hid.view(self.num_layers, 2, enc_hid.size(1), -1)
        return torch.tanh(self.bridge(torch.cat((hidden[-1][0], hidden[-1][1]), dim=1))).unsqueeze(0).repeat(self.num_layers, 1, 1)
    
    def forward(self, tok, dec_hid, enc_out, meta, glob_ctx):
        embedded = self.dropout(self.embedding(tok.unsqueeze(0)))
        scores = torch.bmm(dec_hid[-1].unsqueeze(0).permute(1, 0, 2), self.attention(enc_out).permute(1, 2, 0))
        context = torch.bmm(F.softmax(scores, dim=2), enc_out.permute(1, 0, 2)).permute(1, 0, 2)
        return self.fc_out(self.rnn(torch.cat((embedded, context, meta.unsqueeze(0), glob_ctx.unsqueeze(0)), dim=2), dec_hid)[0].squeeze(0)), self.rnn(torch.cat((embedded, context, meta.unsqueeze(0), glob_ctx.unsqueeze(0)), dim=2), dec_hid)[1], context.squeeze(1)

class Seq2Seq(nn.Module):
    def __init__(self, vocab_size, num_colors, num_objects, pad_id):
        super().__init__()
        self.encoder = SpatioTemporalEEGEncoder(num_channels=62, enc_hidden=256) 
        self.meta_encoder = MetadataEncoder(num_colors, num_objects)
        self.decoder = Decoder(vocab_size, 256, 256, 256, 64, 2, pad_id, 0.0)
        self.meta_head = nn.Sequential(nn.Linear(512, 256), nn.ReLU(), nn.LayerNorm(256), nn.Dropout(0.0), nn.Linear(256, num_colors + num_objects))

# ==================================================================================
# 3. BEAM SEARCH DECODER (Single Sample)
# ==================================================================================

def beam_search_decoder(model, eeg, meta, adj_matrix, beam_width=3):
    model.eval()
    with torch.no_grad():
        eeg = eeg.unsqueeze(0).to(device)
        meta = meta.unsqueeze(0).to(device)
        
        enc_out, enc_hid = model.encoder(eeg, adj_matrix)
        meta_feat = model.meta_encoder(meta) 
        dec_hid = model.decoder.init_hidden(enc_hid)
        glob_ctx = torch.cat((enc_hid.view(2, 2, 1, -1)[-1][0], enc_hid.view(2, 2, 1, -1)[-1][1]), dim=1)
        
        beams = [(0.0, SOS_ID, dec_hid, [])]
        
        for _ in range(30):
            candidates = []
            for score, inp, hid, seq in beams:
                if len(seq) > 0 and seq[-1] == EOS_ID:
                    candidates.append((score, inp, hid, seq)); continue
                
                out, new_hid, _ = model.decoder(torch.tensor([inp], device=device), hid, enc_out, meta_feat, glob_ctx)
                log_probs = F.log_softmax(out, dim=-1).squeeze(0)
                topk_probs, topk_ids = log_probs.topk(beam_width)
                
                for k in range(beam_width):
                    candidates.append((score + topk_probs[k].item(), topk_ids[k].item(), new_hid, seq + [topk_ids[k].item()]))
            
            beams = sorted(candidates, key=lambda x: x[0], reverse=True)[:beam_width]
            if all(seq[-1] == EOS_ID for _, _, _, seq in beams if len(seq) > 0): break

        best_seq = beams[0][3][:-1] if beams[0][3] and beams[0][3][-1] == EOS_ID else beams[0][3]
        return tokenizer.decode(best_seq, skip_special_tokens=True)

# ==================================================================================
# 4. MAIN EVALUATION
# ==================================================================================
if __name__ == "__main__":
    # 1. Load Model
    model = Seq2Seq(TEXT_VOCAB_SIZE, NUM_COLORS, NUM_OBJECTS, PAD_ID).to(device)
    try:
        model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, map_location=device))
        print(f"Loaded weights from {MODEL_WEIGHTS_PATH}")
    except:
        print(f"Error: Could not find {MODEL_WEIGHTS_PATH}.")
        exit()

    # 2. Setup Adjacency Matrix
    adj_matrix = torch.ones((62, 62)).to(device) / 62.0 

    # 3. Setup Metrics
    bleu = evaluate.load("bleu")
    rouge = evaluate.load("rouge")

    # 4. Process Data
    print("Processing Dataset...")
    predictions = []
    references = []

    with h5py.File(H5_FILE_PATH, 'r') as f:
        total_samples = f['eeg'].shape[0]
        # Recalculate Test Indices (Every 5th sample starting at index 4)
        test_indices = list(range(total_samples))
        print(f"Total Test Samples: {len(test_indices)}")

        for idx in tqdm(test_indices, desc="Evaluating"):
            # Load Data
            eeg = torch.from_numpy(f['eeg'][idx].astype(np.float32)).to(device)
            meta = torch.from_numpy(f['metadata'][idx].astype(np.float32)).to(device)
            txt_ids = f['input_ids'][idx].astype(np.int64)
            gt_text = tokenizer.decode(txt_ids, skip_special_tokens=True)

            # Generate
            pred_text = beam_search_decoder(model, eeg, meta, adj_matrix)
            
            predictions.append(pred_text)
            references.append(gt_text)

    # 5. Compute & Print Scores
    print("\n" + "="*40)
    print(f"FINAL RESULTS ON TEST SET ({len(test_indices)} Samples)")
    print("="*40)

    # BLEU
    # evaluate expects references as list of lists for BLEU: [[ref1], [ref2]]
    bleu_score = bleu.compute(predictions=predictions, references=[[r] for r in references])
    print(f"BLEU:      {bleu_score['bleu']:.4f}")

    # ROUGE
    rouge_score = rouge.compute(predictions=predictions, references=references)
    print(f"ROUGE-1:   {rouge_score['rouge1']:.4f}")
    print(f"ROUGE-2:   {rouge_score['rouge2']:.4f}")
    print(f"ROUGE-L:   {rouge_score['rougeL']:.4f}")
    print("="*40)

Running Evaluation on: cuda
Loaded weights from static-graph-phase-2.pt
Processing Dataset...
Total Test Samples: 28000


Evaluating:   0%|          | 0/28000 [00:00<?, ?it/s]


FINAL RESULTS ON TEST SET (28000 Samples)
BLEU:      0.2104
ROUGE-1:   0.4519
ROUGE-2:   0.2581
ROUGE-L:   0.4447


Running Evaluation on: cuda
Loaded weights from static-graph-phase-2.pt
Processing Dataset...
Total Test Samples: 28000
Evaluating: 100%
 28000/28000 [07:57<00:00, 60.32it/s]

========================================
FINAL RESULTS ON TEST SET (28000 Samples)
========================================
BLEU:      0.2104
ROUGE-1:   0.4519
ROUGE-2:   0.2581
ROUGE-L:   0.4447
========================================

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import h5py
from torch.utils.data import Dataset, DataLoader, Subset
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer
from tqdm.auto import tqdm
import evaluate  # pip install evaluate rouge_score
import os

# ==================================================================================
# 1. CONFIGURATION
# ==================================================================================
MODEL_WEIGHTS_PATH = "static-graph-phase-2.pt"  # Or "static-graph-phase-1.pt" depending on your save
H5_FILE_PATH = "/home/poorna/data/eeg_dataset_1400_multilabel.h5"
LOCAL_MODEL_PATH = "/home/poorna/models/bert-base-uncased"

# Use GPU if available for speed, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running Evaluation on: {device}")

# Tokenizer
try:
    tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)
except:
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

PAD_ID = tokenizer.pad_token_id
SOS_ID = tokenizer.cls_token_id
EOS_ID = tokenizer.sep_token_id
TEXT_VOCAB_SIZE = tokenizer.vocab_size
NUM_COLORS = 9       
NUM_OBJECTS = 6      

# ==================================================================================
# 2. MODEL DEFINITIONS (Dense GCN - Must Match Training)
# ==================================================================================

class DenseGCNLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)

    def forward(self, x, adj):
        x = self.linear(x) 
        return torch.einsum('nm, btni -> btmi', adj, x)

class SpatioTemporalEEGEncoder(nn.Module):
    def __init__(self, num_channels=62, enc_hidden=256, num_layers=2, dropout=0.2):
        super().__init__()
        self.num_channels = num_channels
        self.gcn1 = DenseGCNLayer(1, enc_hidden) 
        self.gcn2 = DenseGCNLayer(enc_hidden, enc_hidden)
        self.rnn = nn.GRU(enc_hidden, enc_hidden, num_layers, bidirectional=True, dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, eeg, adj_matrix):
        # eeg: [Batch, Channels, Time] -> [Batch, Time, Channels, 1]
        x = eeg.permute(0, 2, 1).unsqueeze(-1) 
        x = F.relu(self.gcn1(x, adj_matrix))
        x = self.dropout(x)
        x = F.relu(self.gcn2(x, adj_matrix)) 
        x = torch.mean(x, dim=2) # Mean Pool over Channels
        encoder_outputs, encoder_hidden = self.rnn(x)
        return encoder_outputs.permute(1, 0, 2), encoder_hidden

class MetadataEncoder(nn.Module):
    def __init__(self, num_colors, num_objects):
        super().__init__()
        self.color_p = nn.Sequential(nn.Linear(num_colors, 64), nn.ReLU(), nn.Linear(64, 32))
        self.obj_p = nn.Sequential(nn.Linear(num_objects, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, 32))
        self.output_dim = 64
    def forward(self, m): return torch.cat([self.color_p(m[:,:9]), self.obj_p(m[:,9:])], dim=1)

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, enc_hidden, dec_hidden, meta_dim, num_layers, pad_id, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.attention = nn.Linear(enc_hidden * 2, dec_hidden)
        self.rnn = nn.GRU(emb_dim + enc_hidden * 2 + meta_dim + enc_hidden * 2, dec_hidden, num_layers, dropout=dropout)
        self.fc_out = nn.Linear(dec_hidden, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.bridge = nn.Linear(enc_hidden * 2, dec_hidden)
        self.num_layers = num_layers
    
    def init_hidden(self, enc_hid):
        hidden = enc_hid.view(self.num_layers, 2, enc_hid.size(1), -1)
        return torch.tanh(self.bridge(torch.cat((hidden[-1][0], hidden[-1][1]), dim=1))).unsqueeze(0).repeat(self.num_layers, 1, 1)
    
    def forward(self, tok, dec_hid, enc_out, meta, glob_ctx):
        embedded = self.dropout(self.embedding(tok.unsqueeze(0)))
        scores = torch.bmm(dec_hid[-1].unsqueeze(0).permute(1, 0, 2), self.attention(enc_out).permute(1, 2, 0))
        context = torch.bmm(F.softmax(scores, dim=2), enc_out.permute(1, 0, 2)).permute(1, 0, 2)
        return self.fc_out(self.rnn(torch.cat((embedded, context, meta.unsqueeze(0), glob_ctx.unsqueeze(0)), dim=2), dec_hid)[0].squeeze(0)), self.rnn(torch.cat((embedded, context, meta.unsqueeze(0), glob_ctx.unsqueeze(0)), dim=2), dec_hid)[1], context.squeeze(1)

class Seq2Seq(nn.Module):
    def __init__(self, vocab_size, num_colors, num_objects, pad_id):
        super().__init__()
        self.encoder = SpatioTemporalEEGEncoder(num_channels=62, enc_hidden=256) 
        self.meta_encoder = MetadataEncoder(num_colors, num_objects)
        self.decoder = Decoder(vocab_size, 256, 256, 256, 64, 2, pad_id, 0.0)
        self.meta_head = nn.Sequential(nn.Linear(512, 256), nn.ReLU(), nn.LayerNorm(256), nn.Dropout(0.0), nn.Linear(256, num_colors + num_objects))

# ==================================================================================
# 3. BEAM SEARCH DECODER (Single Sample)
# ==================================================================================

def beam_search_decoder(model, eeg, meta, adj_matrix, beam_width=3):
    model.eval()
    with torch.no_grad():
        eeg = eeg.unsqueeze(0).to(device)
        meta = meta.unsqueeze(0).to(device)
        
        enc_out, enc_hid = model.encoder(eeg, adj_matrix)
        meta_feat = model.meta_encoder(meta) 
        dec_hid = model.decoder.init_hidden(enc_hid)
        glob_ctx = torch.cat((enc_hid.view(2, 2, 1, -1)[-1][0], enc_hid.view(2, 2, 1, -1)[-1][1]), dim=1)
        
        beams = [(0.0, SOS_ID, dec_hid, [])]
        
        for _ in range(30):
            candidates = []
            for score, inp, hid, seq in beams:
                if len(seq) > 0 and seq[-1] == EOS_ID:
                    candidates.append((score, inp, hid, seq)); continue
                
                out, new_hid, _ = model.decoder(torch.tensor([inp], device=device), hid, enc_out, meta_feat, glob_ctx)
                log_probs = F.log_softmax(out, dim=-1).squeeze(0)
                topk_probs, topk_ids = log_probs.topk(beam_width)
                
                for k in range(beam_width):
                    candidates.append((score + topk_probs[k].item(), topk_ids[k].item(), new_hid, seq + [topk_ids[k].item()]))
            
            beams = sorted(candidates, key=lambda x: x[0], reverse=True)[:beam_width]
            if all(seq[-1] == EOS_ID for _, _, _, seq in beams if len(seq) > 0): break

        best_seq = beams[0][3][:-1] if beams[0][3] and beams[0][3][-1] == EOS_ID else beams[0][3]
        return tokenizer.decode(best_seq, skip_special_tokens=True)

# ==================================================================================
# 4. MAIN EVALUATION
# ==================================================================================
if __name__ == "__main__":
    # 1. Load Model
    model = Seq2Seq(TEXT_VOCAB_SIZE, NUM_COLORS, NUM_OBJECTS, PAD_ID).to(device)
    try:
        model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, map_location=device))
        print(f"Loaded weights from {MODEL_WEIGHTS_PATH}")
    except:
        print(f"Error: Could not find {MODEL_WEIGHTS_PATH}.")
        exit()

    # 2. Setup Adjacency Matrix
    adj_matrix = torch.ones((62, 62)).to(device) / 62.0 

    # 3. Setup Metrics
    bleu = evaluate.load("bleu")
    rouge = evaluate.load("rouge")

    # 4. Process Data
    print("Processing Dataset...")
    predictions = []
    references = []

    with h5py.File(H5_FILE_PATH, 'r') as f:
        total_samples = f['eeg'].shape[0]
        # Recalculate Test Indices (Every 5th sample starting at index 4)
        test_indices = list(range(4,total_samples,5))[:20]
        print(f"Total Test Samples: {len(test_indices)}")

        for idx in tqdm(test_indices, desc="Evaluating"):
            # Load Data
            eeg = torch.from_numpy(f['eeg'][idx].astype(np.float32)).to(device)
            meta = torch.from_numpy(f['metadata'][idx].astype(np.float32)).to(device)
            txt_ids = f['input_ids'][idx].astype(np.int64)
            gt_text = tokenizer.decode(txt_ids, skip_special_tokens=True)

            # Generate
            pred_text = beam_search_decoder(model, eeg, meta, adj_matrix)
            print(f"Ground truth text (Sample {idx}) : {gt_text}")
            print(f"Predicted truth text: {pred_text}")
            print("--------------------------------------------")
            predictions.append(pred_text)
            references.append(gt_text)

    # 5. Compute & Print Scores
    print("\n" + "="*40)
    print(f"FINAL RESULTS ON TEST SET ")
    print("="*40)

    # BLEU
    # evaluate expects references as list of lists for BLEU: [[ref1], [ref2]]
    bleu_score = bleu.compute(predictions=predictions, references=[[r] for r in references])
    print(f"BLEU:      {bleu_score['bleu']:.4f}")

    # ROUGE
    rouge_score = rouge.compute(predictions=predictions, references=references)
    print(f"ROUGE-1:   {rouge_score['rouge1']:.4f}")
    print(f"ROUGE-2:   {rouge_score['rouge2']:.4f}")
    print(f"ROUGE-L:   {rouge_score['rougeL']:.4f}")
    print("="*40)

Running Evaluation on: cuda
Loaded weights from static-graph-phase-2.pt
Processing Dataset...
Total Test Samples: 20


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

Ground truth text (Sample 4) : a city at night with buildings lit up
Predicted truth text: a city with tall buildings
--------------------------------------------
Ground truth text (Sample 9) : a beach with a cloudy sky and a beach
Predicted truth text: a waterfall in the middle of a river
--------------------------------------------
Ground truth text (Sample 14) : a blue jellyfish in the dark
Predicted truth text: fireworks in the night sky
--------------------------------------------
Ground truth text (Sample 19) : four small rabbits in a bowl
Predicted truth text: a cat with yellow eyes
--------------------------------------------
Ground truth text (Sample 24) : a man is snowboarding on a snowy mountain
Predicted truth text: a man playing a guitar
--------------------------------------------
Ground truth text (Sample 29) : a forest with trees and bushes in the fore
Predicted truth text: a waterfall in the middle of a river
--------------------------------------------
Ground truth te